# Splite — synthetic model capture → trained Godot cloud

Select **Runtime → Change runtime type → GPU** before running. Installation, CUDA forward/backward, and a real calibration fit/evaluation/export/import have passed on T4. Garden asset quality is still being evaluated. Colab availability and runtime packages vary.

On the Mac, render a source asset, then run `python3 tools/pipeline/bundle_capture.py --data assets/captures/tree_hero --out /tmp/tree_hero.zip`. Upload that archive below. It contains the exact working pipeline, so uncommitted fixes travel with the capture. No repository credentials or Drive mount needed.

Start with one static prop and 64 views at 256–512 pixels. Keep the full PLY and compare it with the reduced `.tres` in Godot; the current runtime draws isotropic splats and may lose fidelity.

Use runtime version **2026.07 (Python 3.12)** for this candidate stack. The latest Python 3.13 runtime was tested and cannot resolve the Open3D dependency. Changing runtime versions restarts the runtime and may clear its files.

The first CUDA extension build can take over 30 minutes on a standard T4 host. Keep `MAX_JOBS=2` to limit host memory use; later runs on the same runtime reuse the compiled extension.


In [ ]:
import subprocess, sys
from pathlib import Path
assert sys.version_info < (3, 13), 'Select Runtime > Change runtime type > Runtime version 2026.07, keeping T4 GPU.'
print(sys.version)
print(subprocess.check_output(['nvidia-smi'], text=True))
r = subprocess.run([sys.executable, '-m', 'pip', 'install', 'nerfstudio==1.1.5', 'gsplat==1.4.0', 'Pillow>=10.3', 'ninja'], capture_output=True, text=True)
Path('/content/splite-install.log').write_text(r.stdout + r.stderr)
print((r.stdout + r.stderr)[-3000:])
r.check_returncode()


In [ ]:
import torch, importlib.metadata, json
from pathlib import Path
assert torch.cuda.is_available(), 'Select a GPU runtime'
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)
subprocess.run(['ns-train', 'splatfacto', '--help'], check=True, stdout=subprocess.DEVNULL)
subprocess.run(['nvcc', '--version'], check=True)


In [ ]:
"""Check the real gsplat CUDA forward/backward path without a source dataset."""
import os

# Limit first-build RAM use on small Colab hosts.
os.environ.setdefault('MAX_JOBS', '2')

import torch
from gsplat import rasterization


def main():
    assert torch.cuda.is_available(), 'CUDA unavailable'
    device = 'cuda'
    means = torch.tensor([[0.0, 0.0, 3.0], [0.2, 0.1, 3.2]], device=device, requires_grad=True)
    quats = torch.tensor([[1., 0., 0., 0.]] * 2, device=device, requires_grad=True)
    scales = torch.full((2, 3), 0.15, device=device, requires_grad=True)
    opacity = torch.full((2,), 0.8, device=device, requires_grad=True)
    colors = torch.tensor([[0.8, 0.2, 0.1], [0.1, 0.7, 0.3]], device=device, requires_grad=True)
    view = torch.eye(4, device=device)[None]
    intrinsics = torch.tensor([[[40., 0., 16.], [0., 40., 16.], [0., 0., 1.]]], device=device)
    rgb, alpha, _ = rasterization(means, quats, scales, opacity, colors, view, intrinsics,
                                  width=32, height=32, packed=False, sh_degree=None)
    loss = rgb.square().mean() + alpha.mean()
    loss.backward()
    torch.cuda.synchronize()
    assert torch.isfinite(rgb).all() and alpha.max() > 0
    for name, parameter in [('means', means), ('quats', quats), ('scales', scales), ('opacity', opacity), ('colors', colors)]:
        assert parameter.grad is not None and torch.isfinite(parameter.grad).all(), name
    assert colors.grad.abs().sum() > 0 and scales.grad.abs().sum() > 0
    print('PASS: gsplat CUDA forward/backward on', torch.cuda.get_device_name(0))


if __name__ == '__main__':
    main()


Upload the bundle. Capture extraction rejects paths outside its fresh temporary directory. Re-upload into a new directory to retry training; previous checkpoints remain available for inspection.

In [ ]:
from google.colab import files
import tempfile, zipfile, io
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one capture ZIP'
work = Path(tempfile.mkdtemp(prefix='splite-', dir='/content'))
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for item in archive.infolist():
        destination = (work / item.filename).resolve()
        assert destination.is_relative_to(work), 'Unsafe archive path'
        assert ((item.external_attr >> 16) & 0o170000) != 0o120000, 'No symbolic links'
    archive.extractall(work)
assert (work / 'capture/recipe.json').exists()
print(json.loads((work / 'capture/recipe.json').read_text()))


In [ ]:
asset_id = 'tree_hero'  # Change for the asset being fitted.
iterations = 7000
import_budget = 600  # Import reduction, NOT a training population constraint.
assert asset_id.replace('_', '').isalnum()
results = work / 'results'
results.mkdir(exist_ok=True)
(results / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
(results / 'gpu.txt').write_text(subprocess.check_output(['nvidia-smi'], text=True))
(results / 'cuda.txt').write_text(subprocess.check_output(['nvcc', '--version'], text=True))
subprocess.run([sys.executable, str(work / 'pipeline/train/train_cloud.py'),
                '--data', str(work / 'capture'), '--out', str(results / (asset_id + '.ply')),
                '--iters', str(iterations), '--budget', str(import_budget)], check=True)


Download results before disconnecting. The archive includes the source-frame PLY, Godot `.tres`, evaluation metrics, exact environment, checkpoint, config, and run manifest. Checkpoints can be large. Copy approved `.tres` files to `assets/clouds/` on the Mac; preserve source attribution before integrating them. A successful fit is not an automatic visual-quality pass.

In [ ]:
import shutil
archive = shutil.make_archive(str(work / (asset_id + '-results')), 'zip', results)
files.download(archive)
